# TRIGA FE101 B1 — five-ring radial depletion preparation

Separate copy derived from `triga_single_FE101_B1_1965_2026_CLEAN.ipynb`.
The original notebook is not modified. Full 1965 core: 61 FE101; only five
distinct equal-area B1 materials deplete, other 60 FE101 remain frozen.

**Reference geometry is physical B:** fuel R=1.791 cm; near-vacuum He4 gap
1.791–1.804 cm; Al clad 1.804–1.880 cm in the active segment. Legacy A is
retained only in ordinary, unchanged FE101, not as the target baseline.

Transport and depletion are disabled by default. Preparation performs only
Python object construction, deterministic geometry checks and initial XML export.


In [ ]:
import json
import re
from pathlib import Path

RUN_DEPLETION = False
RUN_POSTPROCESS = False
EXPORT_PREPARED_INPUTS = True
CASE_DIR = Path(globals().get("FE101_CASE_DIR", Path.cwd()))
if not (CASE_DIR/"radial_geometry.py").is_file():
    CASE_DIR = next(parent/"z3st/cases/verification/neutronics/fe101_radial_depletion"
                    for parent in [Path.cwd(), *Path.cwd().parents]
                    if (parent/"z3st/cases/verification/neutronics/fe101_radial_depletion/radial_geometry.py").is_file())
SOURCE_MODEL = CASE_DIR.parent/"fe101_geometry_static/source_model.py"
OUTPUT_ROOT = Path(globals().get("FE101_OUTPUT_ROOT", CASE_DIR/"preparation"))
PREPARED_DIR = OUTPUT_ROOT/"prepared_inputs"
RUN_DIR = OUTPUT_ROOT/"run_3500h_ED"


## Original CLEAN transport definitions — unchanged core, materials and rods


In [ ]:
# Original CLEAN cell 1 (zero-based)
import os
import numpy as np
import openmc
import openmc.deplete
import pandas as pd
from pathlib import Path



In [ ]:
# Original CLEAN cell 3 (zero-based)
# Nomenclature for thermal libraries: https://docs.openmc.org/en/stable/_modules/openmc/data/thermal.html

T_ref_w = 294 # (K)
T_ref_f = 294 # (K)
T_ref_struct = 294 # (K)

# Water
Water = openmc.Material(name='Water')
Water.set_density('g/cm3',0.9985)
Water.add_nuclide('H1', 2, 'ao')
Water.add_nuclide('O16', 1, 'ao')
Water.add_s_alpha_beta('c_H_in_H2O')
Water.temperature = T_ref_w # (K)

# Fuel 101 @ Pavia 
Fuel101 = openmc.Material(name='Fuel 101')
Fuel101.set_density('g/cm3', 6.3)
Fuel101.add_nuclide('U235', 1.58, 'wo')
Fuel101.add_nuclide('U238', 6.42, 'wo')
Fuel101.add_nuclide('H1', 1, 'wo')
Fuel101.add_nuclide('Zr90', 46.1, 'wo')
Fuel101.add_nuclide('Zr91', 10.2, 'wo')
Fuel101.add_nuclide('Zr92', 15.7, 'wo')
Fuel101.add_nuclide('Zr94', 16.3, 'wo')
Fuel101.add_nuclide('Zr96', 2.68, 'wo')
Fuel101.add_s_alpha_beta('c_H_in_ZrH')
#Fuel101.add_s_alpha_beta('c_Zr_in_ZrH')
Fuel101.temperature = T_ref_f

# Fuel 103 @ Pavia 
Fuel103 = openmc.Material(name='Fuel 103')
Fuel103.set_density('g/cm3', 5.95)
Fuel103.add_nuclide('U235', 1.58, 'wo')
Fuel103.add_nuclide('U238', 6.42, 'wo')
Fuel103.add_nuclide('H1', 1.59, 'wo')
Fuel103.add_nuclide('Zr90', 45.8, 'wo')
Fuel103.add_nuclide('Zr91', 10.1, 'wo')
Fuel103.add_nuclide('Zr92', 15.6, 'wo')
Fuel103.add_nuclide('Zr94', 16.2, 'wo')
Fuel103.add_nuclide('Zr96', 2.66, 'wo')
Fuel103.add_s_alpha_beta('c_H_in_ZrH')
#Fuel103.add_s_alpha_beta('c_Zr_in_ZrH')
Fuel103.temperature= T_ref_f

# Graphite
Graphite = openmc.Material(name='Graphite')
Graphite.set_density('g/cm3', 1.675)
Graphite.add_element('C', 0.99, 'wo')
Graphite.add_nuclide('Fe54', 0.00045164, 'wo')
Graphite.add_nuclide('Fe56', 0.00735212, 'wo')
Graphite.add_nuclide('Fe57', 0.00017283, 'wo')
Graphite.add_nuclide('Fe58', 0.00002340, 'wo')
Graphite.add_nuclide('O16', 0.0013, 'wo')
Graphite.add_nuclide('Ti46', 7.92E-05, 'wo')
Graphite.add_nuclide('Ti47', 7.30E-05, 'wo')
Graphite.add_nuclide('Ti48', 7.38E-04, 'wo')
Graphite.add_nuclide('Ti49', 5.53E-05, 'wo')
Graphite.add_nuclide('Ti50', 5.40E-05, 'wo')
Graphite.add_nuclide('Na23', 0.0001, 'wo')
Graphite.add_nuclide('Mg24', 7.795E-05, 'wo')
Graphite.add_nuclide('Mg25', 1.028E-05, 'wo')
Graphite.add_nuclide('Mg26', 1.177E-05, 'wo')
Graphite.add_nuclide('Cu63', 6.85E-04, 'wo')
Graphite.add_nuclide('Cu65', 3.15E-04, 'wo')
Graphite.add_nuclide('Co59', 0.00005, 'wo')
Graphite.add_nuclide('H1', 0.000037, 'wo')
Graphite.add_nuclide('F19', 0.000025, 'wo')
Graphite.add_element('Cl', 0.000025, 'wo')
Graphite.add_nuclide('B11', 0.0000005, 'wo')
Graphite.add_s_alpha_beta('c_Graphite')
Graphite.temperature = T_ref_struct

# Cladding Al (from Chiesa PhD Thesis) 
CladdingAl = openmc.Material(name='Cladding Al') 
CladdingAl.set_density('g/cm3' , 2.713) 
CladdingAl.add_nuclide('Al27', 0.9957, 'wo')
CladdingAl.add_nuclide('Cu63', 0.0006917, 'wo')
CladdingAl.add_nuclide('Cu65', 0.0003083, 'wo')
CladdingAl.temperature = T_ref_struct

# Cladding SS
CladdingSS = openmc.Material(name='Cladding SS') 
CladdingSS.set_density('g/cm3' , 8.03) 
CladdingSS.add_nuclide('Fe54' , 0.039980, 'wo')
CladdingSS.add_nuclide('Fe56' , 0.650820, 'wo')
CladdingSS.add_nuclide('Fe57' , 0.015299, 'wo')
CladdingSS.add_nuclide('Fe58' , 0.002072, 'wo')
CladdingSS.add_nuclide('Cr50' , 7.72E-03, 'wo')
CladdingSS.add_nuclide('Cr52' , 1.49E-01, 'wo')
CladdingSS.add_nuclide('Cr53' , 1.69E-02, 'wo')
CladdingSS.add_nuclide('Cr54' , 4.19E-03, 'wo')
CladdingSS.add_nuclide('Ni58' , 7.42E-02, 'wo')
CladdingSS.add_nuclide('Ni60' , 2.86E-02, 'wo')
CladdingSS.add_nuclide('Ni61' , 1.24E-03, 'wo')
CladdingSS.add_nuclide('Ni62' , 3.95E-03, 'wo')
CladdingSS.add_nuclide('Ni64' , 1.01E-03, 'wo')
CladdingSS.add_nuclide('Si28' , 5.02E-03, 'wo')
CladdingSS.add_nuclide('Si29' , 2.55E-04, 'wo')
CladdingSS.add_nuclide('Si30' , 1.68E-04, 'wo')
CladdingSS.temperature = T_ref_struct

# Cladding for Control Rods (Aluminum)
CladdingCR = openmc.Material(name='Cladding CR') 
CladdingCR.set_density('g/cm3' , 2.71)
CladdingCR.add_nuclide('Al27', 0.9795, 'wo')
CladdingCR.add_nuclide('Cu63', 0.00172925, 'wo')
CladdingCR.add_nuclide('Cu65', 0.00077075, 'wo')
CladdingCR.add_nuclide('Cr52', 0.0020, 'wo')
CladdingCR.add_nuclide('Mg24' , 7.80E-03 , 'wo')
CladdingCR.add_nuclide('Mg25' , 1.03E-03 , 'wo')
CladdingCR.add_nuclide('Mg26' , 1.18E-03 , 'wo')
CladdingCR.add_nuclide('Si28', 0.006, 'wo')
CladdingCR.temperature = T_ref_struct  

# Zirconium thin rod
Zirconium = openmc.Material(name='Zirconium')
Zirconium.set_density('g/cm3', 6.52)
Zirconium.add_nuclide('Zr90', 0.5145, 'ao')
Zirconium.add_nuclide('Zr91', 0.1122, 'ao')
Zirconium.add_nuclide('Zr92', 0.1715, 'ao')
Zirconium.add_nuclide('Zr94', 0.1738, 'ao')
Zirconium.add_nuclide('Zr96', 0.028, 'ao')
Zirconium.temperature = T_ref_f 

# Poison disk
Samarium = openmc.Material(name='Sm poison disk')
Samarium.set_density('g/cm3', 2.4203) 
Samarium.add_nuclide('Sm147', 0.001035, 'wo')
Samarium.add_nuclide('Sm148', 0.000782, 'wo')
Samarium.add_nuclide('Sm149', 0.000968, 'wo')
Samarium.add_nuclide('Sm150', 0.000520, 'wo')
Samarium.add_nuclide('Sm152', 0.001911, 'wo')
Samarium.add_nuclide('Sm154', 0.001647, 'wo')
Samarium.add_nuclide('O16', 0.468019, 'wo')
Samarium.add_nuclide('Al27', 0.524910, 'wo')
Samarium.temperature = T_ref_f

# Vacuum
Vacuum = openmc.Material(name='Vacuum')
Vacuum.set_density('g/cm3', 0.0012041)
Vacuum.add_element('C', 0.000125, 'wo')
Vacuum.add_nuclide('N14', 0.6869, 'wo')
Vacuum.add_nuclide('O16', 0.301248, 'wo')
Vacuum.temperature = 294

# SHIM - REG
ControlRodB4C = openmc.Material(name='Control rods (SHIM, REG)')
ControlRodB4C.set_density('g/cm3' , 2.52)
ControlRodB4C.add_nuclide('B11' , 64.2, 'ao')
ControlRodB4C.add_nuclide('B10' , 15.8, 'ao')
ControlRodB4C.add_element('C' , 20, 'ao')
ControlRodB4C.add_s_alpha_beta('c_Graphite')
ControlRodB4C.temperature = T_ref_struct

# TRANS
ControlRodBorated = openmc.Material(name='Control rods (TRANS)')
ControlRodBorated.set_density('g/cm3' , 2.23)
ControlRodBorated.add_nuclide('B11' , 21.7, 'ao')
ControlRodBorated.add_nuclide('B10' , 5.35, 'ao')
ControlRodBorated.add_element('C' , 72.3, 'ao')
ControlRodBorated.add_s_alpha_beta('c_Graphite')
ControlRodBorated.temperature = T_ref_struct


# Instantiate a Materials collection and export to xml
materials_file = openmc.Materials([CladdingAl, CladdingSS, CladdingCR, Graphite, Fuel101, Fuel103, Water, Vacuum, ControlRodB4C, ControlRodBorated, Samarium, Zirconium ])




In [ ]:
# Original CLEAN cell 5 (zero-based)
# Control rod position: 0 = all in, 47 = all out
z0_reg_value = 45 # (cm)
z0_shim_value = 45 # (cm)
z0_trans_value = 45 # (cm)



In [ ]:
# Original CLEAN cell 6 (zero-based)
# Planes, cylinders and stuff

r_fuel_101 = openmc.ZCylinder(r=1.791) #fuel radius
r_clad_101 = openmc.ZCylinder(r=1.88)  #cladding radius
bot_active_101 = openmc.ZPlane(z0=10.20) #start active region 101
top_active_101 = openmc.ZPlane(z0=45.76) #end active region 101
Sm_bot_101 = openmc.ZPlane(z0=10.07) # samarium disk bot end 
Sm_top_101 = openmc.ZPlane(z0=45.89) # samarium disk top end 
reflector_bot_101 = openmc.ZPlane(z0=0) #bottom 101 reflector
reflector_top_101 = openmc.ZPlane(z0=55.83) #top 101 reflector
clad_bot_101 = openmc.ZPlane(z0=-1.27) #end bottom cladding 101
clad_top_101 = openmc.ZPlane(z0=57.1) #end top cladding 101
plug_top_101 = openmc.ZPlane(z0=64.26) #end plug 101
plug_radius_101 = openmc.ZCylinder(r=0.395)


# Geometry definitions for the Fuel Rod 103
r_fuel_inner_103 = openmc.ZCylinder(r=0.285) #fuel radius
r_fuel_outer_103 = openmc.ZCylinder(r=1.82) #fuel radius
r_clad_103 = openmc.ZCylinder(r=1.88)  #cladding radius
bot_active_103 = openmc.ZPlane(z0=8.81) #start active region 103
top_active_103 = openmc.ZPlane(z0=46.91) #end active region 103
reflector_bot_103 = openmc.ZPlane(z0=0) #bottom 103 reflector
reflector_top_103 = openmc.ZPlane(z0=55.65) #top 103 reflector
clad_bot_103 = openmc.ZPlane(z0=-1.39) #end bottom cladding 103
clad_top_103 = openmc.ZPlane(z0=57.31) #end top cladding 103
plug_top_103 = openmc.ZPlane(z0=64.47) #end plug 103
plug_radius_103 = openmc.ZCylinder(r=0.395)

# Geometry for dummy element (graphite)
r_dummy_graphite = openmc.ZCylinder(r=1.791)
r_dummy_clad = openmc.ZCylinder(r=1.88)
dummy_bot =  openmc.ZPlane(z0=0) #bottom 101 reflector
dummy_top = openmc.ZPlane(z0=55.83) #top 101 reflector
dummy_bot_clad = openmc.ZPlane(z0=-1.27) #end bottom cladding 101
dummy_top_clad = openmc.ZPlane(z0=57.1) #end top cladding 101

# Geometry for central channel (with Al case for void feedback measurements)
r_central_channel = openmc.ZCylinder(r=1.69)
r_central_channel_clad = openmc.ZCylinder(r=1.905)

r_Al_out = 3/2 # (cm)
r_Al_in = 2.4/2 # (cm)
l_Al_out = 28 # (cm)
l_Al_in = 26.5258238 # (cm)
midplane = 27.86
plane_mat_separation = l_Al_in - 0.01 

s_void_r_out = openmc.ZCylinder(r = r_Al_out)
s_void_r_in = openmc.ZCylinder(r = r_Al_in)
s_void_bot_in = openmc.ZPlane(z0 = midplane - l_Al_in/2)
s_void_bot_out = openmc.ZPlane(z0 = midplane - l_Al_out/2)
s_void_sep = openmc.ZPlane(z0 = midplane - l_Al_in/2 + plane_mat_separation)
s_void_top_in = openmc.ZPlane(z0 = midplane + l_Al_in/2)
s_void_top_out = openmc.ZPlane(z0 = midplane + l_Al_out/2)

# Geometry for rabbit channel 
r_rabbit_inner = openmc.ZCylinder(r=1.804)
r_rabbit_outer = openmc.ZCylinder(r=1.88)

# Geometry for source
r_source = openmc.ZCylinder(r=1.88)

# Geometry for SHIM
r_shim_inner = openmc.ZCylinder(r=1.425)
r_shim_outer = openmc.ZCylinder(r=1.59)
r_shim_bot_plug = openmc.ZCylinder(r=0.795)

# Geometry for REG
r_reg_inner = openmc.ZCylinder(r=0.965)
r_reg_outer = openmc.ZCylinder(r=1.11)
r_reg_bot_plug = openmc.ZCylinder(r=0.555)

# Geometry for TRANS
r_trans_inner = openmc.ZCylinder(r=1.105)
r_trans_outer = openmc.ZCylinder(r=1.27)
r_trans_bot_plug = openmc.ZCylinder(r=0.635)

# CR heigths: length=45.47 cm, the starting position is zero.
#             I define three quantities, z_reg, z_shim and z_trans
#             such that they can be moved

# CR position from their 0
z0_reg= z0_reg_value
z0_shim = z0_shim_value
z0_trans = z0_trans_value

# CALIBRAZIONE: cambiare le altezze delle barre di controllo per calcolare il k
#               (0 = rod tutta inserita, )
# valori reg:  0.27 , 4.57 , 9.85 , 13.77, 17.69, 21.55, 24.77, 27.21, 38.1
# valori shim: 19.92, 19.92, 19.38, 18.45, 16.93, 15.51, 13.93, 12.74, 11.81 
# valori trans: sempre estratta 

# Model offset (z=+10.4 means that the rod is 'all in')
offset= 10.40

z0_reg = z0_reg + offset
z0_shim = z0_shim + offset
z0_trans = z0_trans + offset

bot_reg = openmc.ZPlane(z0=z0_reg) #plane for the bottom reg
bot_shim = openmc.ZPlane(z0=z0_shim) #plane for the bottom shim
bot_trans = openmc.ZPlane(z0=z0_trans) #plane for the bottom trans

bot_reg_clad = openmc.ZPlane(z0=z0_reg-0.8) #plane for the bottom clad
bot_shim_clad = openmc.ZPlane(z0=z0_shim-0.8)
bot_trans_clad = openmc.ZPlane(z0=z0_trans-0.8)

bot_reg_plug = openmc.ZPlane(z0= z0_reg -1.6) #plane for the bottom of plug
bot_shim_plug = openmc.ZPlane(z0= z0_shim -1.6) 
bot_trans_plug = openmc.ZPlane(z0= z0_trans -1.6)

top_reg = openmc.ZPlane(z0=z0_reg+45.47) #plane for the bottom reg
top_shim = openmc.ZPlane(z0=z0_shim+45.47) #plane for the bottom shim
top_trans = openmc.ZPlane(z0=z0_trans+45.47) #plane for the bottom trans

#top plug has width of 2.54 cm
top_clad_reg = openmc.ZPlane(z0=z0_reg+45.47+2.54)
top_clad_shim = openmc.ZPlane(z0=z0_shim+45.47+2.54)
top_clad_trans = openmc.ZPlane(z0=z0_trans+45.47+2.54)

top_CR = openmc.ZPlane(z0=45.47) #control rod length from 0
start_position_CR = openmc.ZPlane(z0=0) #zero position for CR

# Reflector dimensions planes
bot_reflector = openmc.ZPlane(z0=1.27) #start of side reflector
top_reflector = openmc.ZPlane(z0=57.17) #end of side reflector

bot_clad_reflector = openmc.ZPlane(z0=0.0) #start of side reflector's clad
top_clad_reflector = openmc.ZPlane(z0=59.04) #end of side reflector's clad

#Grid at the top of the core
top_grid_bot = openmc.ZPlane(z0=59.04)
top_grid_top = openmc.ZPlane(z0=60.94)

# Grid at the bottom of core
bottom_grid_bot = openmc.ZPlane(z0=-6.995)
bottom_grid_top = openmc.ZPlane(z0=-5.05) # true


# Void in reflector (lazy susan)
refl_void_inner = openmc.ZCylinder(r=30.5)
refl_void_outer = openmc.ZCylinder(r=37)
refl_void_b = openmc.ZPlane(z0=31.35)

#IRRADIATION CHANNELS
#--> Since OpenMC treat badly rotations, the channels' eqns have been defined as quadric (matlab script)
#    Also, a discriminator plane need to be defined since the cylinder is bi-directional. 
#    Finally, a "filling" graphite region need to be defined to fill the water gap.

channel_1_discriminator = openmc.YPlane(y0=0.0)
channel_1_inner = openmc.Quadric(a=0.9963, b=0.0037, c=1, d=-0.1213, e=0, f=0, g=0, h=0, j=-42.7, k=396.5325) 
channel_1_outer = openmc.Quadric(a=0.9963, b=0.0037, c=1, d=-0.1213, e=0, f=0, g=0, h=0, j=-42.7, k=385.2625) 

channel_2_discriminator = openmc.YPlane(y0=0.0)
channel_2_inner = openmc.Quadric(a=361, b=739.5735, c=1100.6, d=-1033.4, e=0, f=0, g=0, h=0, j=-46994, k=436413.1471) 
channel_2_outer = openmc.Quadric(a=361, b=739.5735, c=1100.6, d=-1033.4, e=0, f=0, g=0, h=0, j=-46994, k=424009.68) 

channel_3_discriminator = openmc.Plane(a=-19, b=+27.195, c=0, d=0) #3D plane which cut in half channel 2
channel_3_inner = openmc.Quadric(a=67.1597, b=4.7337, c=71.8933, d=35.6602, e=0, f=0, g=4330.4, h=1149.7, j=-3069.8, k=98312, boundary_type = "transmission")
channel_3_outer = openmc.Quadric(a=67.1597, b=4.7337, c=71.8933, d=35.6602, e=0, f=0, g=4330.4, h=1149.7, j=-3069.8, k=97502, boundary_type = "transmission")


# Core delimitations
core_radius = openmc.ZCylinder(r=22.3, boundary_type = 'transmission') #true
inner_radius_reflector = openmc.ZCylinder(r=23.15)
outer_radius_reflector = openmc.ZCylinder(r=53.35)
s1 = openmc.ZCylinder(r=54.62, boundary_type='vacuum') #cylindrical surface delimiting the core
end_domain_top = openmc.ZPlane(z0=125.7, boundary_type='vacuum')
end_domain_bot = openmc.ZPlane(z0=-50, boundary_type='vacuum') 




In [ ]:
# Original CLEAN cell 8 (zero-based)
# Inline plot

inline_plot = False
cellid = 1



In [ ]:
# Original CLEAN cell 9 (zero-based)
# Fuel element 101
fuel_101_universe = openmc.Universe(name='UZrH Fuel 101 Universe')

fuel_101_cell = openmc.Cell(name='Fuel 101', cell_id = cellid)
fuel_101_cell.fill = Fuel101
fuel_101_cell.region = -r_fuel_101 & +bot_active_101 & -top_active_101

cellid += 1
sm_top_101_cell = openmc.Cell(name='Sm disk top 101', cell_id = cellid)
sm_top_101_cell.fill = Samarium
sm_top_101_cell.region = -r_fuel_101 & +top_active_101 & -Sm_top_101

cellid += 1
graphite_top_101_cell = openmc.Cell(name='graphite top 101', cell_id = cellid)
graphite_top_101_cell.fill = Graphite
graphite_top_101_cell.region = -r_fuel_101 & +Sm_top_101 & -reflector_top_101

cellid += 1
sm_bot_101_cell = openmc.Cell(name='Sm disk bot 101', cell_id = cellid)
sm_bot_101_cell.fill = Samarium
sm_bot_101_cell.region = -r_fuel_101 & +Sm_bot_101 & -bot_active_101

cellid += 1
graphite_bot_101_cell = openmc.Cell(name='graphite bot 101', cell_id = cellid)
graphite_bot_101_cell.fill = Graphite
graphite_bot_101_cell.region = -r_fuel_101 & +reflector_bot_101 & -Sm_bot_101

cellid += 1
clad_101_cell = openmc.Cell(name='Al Cladding', cell_id = cellid)
clad_101_cell.fill = CladdingAl
clad_101_cell.region = -r_clad_101 & +r_fuel_101 & +reflector_bot_101 & -reflector_top_101

cellid += 1
clad_top_101_cell = openmc.Cell(name='Al Cladding top', cell_id = cellid)
clad_top_101_cell.fill = CladdingAl
clad_top_101_cell.region = -r_clad_101 & +reflector_top_101 & -clad_top_101

cellid += 1
clad_bot_101_cell = openmc.Cell(name='Al Cladding bot', cell_id = cellid)
clad_bot_101_cell.fill = CladdingAl
clad_bot_101_cell.region = -r_clad_101 & +clad_bot_101 & -reflector_bot_101

cellid += 1
plug_top_101_cell = openmc.Cell(name='top plug', cell_id = cellid)
plug_top_101_cell.fill = CladdingAl
plug_top_101_cell.region = -plug_radius_101 & +clad_top_101 & -plug_top_101

cellid += 1
plug_bot_101_cell = openmc.Cell(name='bot plug', cell_id = cellid)
plug_bot_101_cell.fill = CladdingAl
plug_bot_101_cell.region = -plug_radius_101 & +bottom_grid_top & -clad_bot_101

cellid += 1
plug_w_101_cell = openmc.Cell(name='top plug water', cell_id = cellid)
plug_w_101_cell.fill = Water
plug_w_101_cell.region = +plug_radius_101 & -r_clad_101 &+clad_top_101 & -plug_top_101

cellid += 1
water_top_fuel_101_cell=openmc.Cell(cell_id = cellid)
water_top_fuel_101_cell.fill = Water
water_top_fuel_101_cell.region = -r_clad_101 & +plug_top_101 &-end_domain_top

cellid += 1
water_bot_fuel_101_cell=openmc.Cell(cell_id = cellid)
water_bot_fuel_101_cell.fill = Water
water_bot_fuel_101_cell.region = -r_clad_101 & +end_domain_bot &-clad_bot_101


fuel_101_universe.add_cell(fuel_101_cell)
fuel_101_universe.add_cell(sm_top_101_cell)
fuel_101_universe.add_cell(graphite_top_101_cell)
fuel_101_universe.add_cell(sm_bot_101_cell)
fuel_101_universe.add_cell(graphite_bot_101_cell)
fuel_101_universe.add_cell(clad_101_cell)
fuel_101_universe.add_cell(clad_top_101_cell)
fuel_101_universe.add_cell(clad_bot_101_cell)
fuel_101_universe.add_cell(plug_top_101_cell)
fuel_101_universe.add_cell(plug_bot_101_cell)
fuel_101_universe.add_cell(plug_w_101_cell)
fuel_101_universe.add_cell(water_top_fuel_101_cell)
fuel_101_universe.add_cell(water_bot_fuel_101_cell)

#if inline_plot:
 #   fuel_101_universe.plot(origin=(0,0,27), width=(10.0, 70.0), pixels=[400, 400], basis='xz', color_by='material')



In [ ]:
# Original CLEAN cell 13 (zero-based)
# Fuel element 103
fuel_103_universe = openmc.Universe(name='UZrH Fuel 103 Universe')

cellid += 1
Zr_rod_103_cell = openmc.Cell(name='Zr thin rod', cell_id = cellid)
Zr_rod_103_cell.fill = Zirconium
Zr_rod_103_cell.region = -r_fuel_inner_103 & +bot_active_103 & -top_active_103

cellid += 1
fuel_103_cell = openmc.Cell(name='Fuel 103', cell_id = cellid)
fuel_103_cell.fill = Fuel103
fuel_103_cell.region = -r_fuel_outer_103 & +r_fuel_inner_103 & +bot_active_103 & -top_active_103

cellid += 1
graphite_top_103_cell = openmc.Cell(name='graphite top 103', cell_id = cellid)
graphite_top_103_cell.fill = Graphite
graphite_top_103_cell.region = -r_fuel_outer_103 & +top_active_103 & -reflector_top_103

cellid += 1
graphite_bot_103_cell = openmc.Cell(name='graphite bot 103', cell_id = cellid)
graphite_bot_103_cell.fill = Graphite
graphite_bot_103_cell.region = -r_fuel_outer_103 & +reflector_bot_103 & -bot_active_103

cellid += 1
clad_103_cell = openmc.Cell(name='SS Cladding 103', cell_id = cellid)
clad_103_cell.fill = CladdingSS
clad_103_cell.region = -r_clad_103 & +r_fuel_outer_103 & +reflector_bot_103 & -reflector_top_103

cellid += 1
clad_top_103_cell = openmc.Cell(name='SS Cladding top 103', cell_id = cellid)
clad_top_103_cell.fill = CladdingSS
clad_top_103_cell.region = -r_clad_103 & +reflector_top_103 & -clad_top_103

cellid += 1
clad_bot_103_cell = openmc.Cell(name='SS Cladding bot 103', cell_id = cellid)
clad_bot_103_cell.fill = CladdingSS
clad_bot_103_cell.region = -r_clad_103 & +clad_bot_103 & -reflector_bot_103

cellid += 1
plug_top_103_cell = openmc.Cell(name='top plug 103', cell_id = cellid)
plug_top_103_cell.fill = CladdingSS
plug_top_103_cell.region = -plug_radius_103 & +clad_top_103 & -plug_top_103

cellid += 1
plug_bot_103_cell = openmc.Cell(name='bot plug 103', cell_id = cellid)
plug_bot_103_cell.fill = CladdingSS
plug_bot_103_cell.region = -plug_radius_103 & +bottom_grid_top & -clad_bot_103

cellid += 1
plug_w_103_cell = openmc.Cell(name='top plug water 103', cell_id = cellid)
plug_w_103_cell.fill = Water
plug_w_103_cell.region = +plug_radius_103 & -r_clad_103 &+clad_top_103 & -plug_top_103

cellid += 1
water_top_fuel_103_cell=openmc.Cell(cell_id = cellid)
water_top_fuel_103_cell.fill = Water
water_top_fuel_103_cell.region = -r_clad_103 & +plug_top_103 &-end_domain_top

cellid += 1
water_bot_fuel_103_cell=openmc.Cell(cell_id = cellid)
water_bot_fuel_103_cell.fill = Water
water_bot_fuel_103_cell.region = -r_clad_103 & ~plug_bot_103_cell.region & +end_domain_bot &-clad_bot_103


fuel_103_universe.add_cell(Zr_rod_103_cell)
fuel_103_universe.add_cell(fuel_103_cell)
fuel_103_universe.add_cell(graphite_top_103_cell)
fuel_103_universe.add_cell(graphite_bot_103_cell)
fuel_103_universe.add_cell(clad_103_cell)
fuel_103_universe.add_cell(clad_top_103_cell)
fuel_103_universe.add_cell(clad_bot_103_cell)
fuel_103_universe.add_cell(plug_bot_103_cell)
fuel_103_universe.add_cell(plug_top_103_cell)
fuel_103_universe.add_cell(plug_w_103_cell)
fuel_103_universe.add_cell(water_top_fuel_103_cell)
fuel_103_universe.add_cell(water_bot_fuel_103_cell)

#if inline_plot:
 #   fuel_103_universe.plot(origin=(0,0,27), width=(10.0, 70.0), pixels=[400, 400], basis='xz', color_by='material')



In [ ]:
# Original CLEAN cell 14 (zero-based)
# Central Channel

# Condition for filling the Al case for void feedback measurements. If void_water = True, fill the case with water. Else, air
void_water = True 

if void_water: 
    FillingMaterial = Water 
else:
    FillingMaterial = Vacuum

central_channel_universe = openmc.Universe(name='CC Universe')

cellid += 1
central_channel_water = openmc.Cell(name='CC water below', cell_id = cellid)
central_channel_water.fill = Water
central_channel_water.region = -r_central_channel_clad & +end_domain_bot & -bottom_grid_bot

cellid += 1
central_channel_clad = openmc.Cell(name='CC clad', cell_id = cellid)
central_channel_clad.fill = CladdingAl
central_channel_clad.region = +r_central_channel & -r_central_channel_clad & +bottom_grid_bot & -end_domain_top

cellid += 1
central_channel_bot_cell = openmc.Cell(name='CC bot', cell_id = cellid)
central_channel_bot_cell.fill = Vacuum
central_channel_bot_cell.region = -r_central_channel & +bottom_grid_bot & -s_void_bot_out

cellid += 1
central_channel_top_cell = openmc.Cell(name='CC top', cell_id = cellid)
central_channel_top_cell.fill = Vacuum
central_channel_top_cell.region = -r_central_channel & +s_void_top_out & -end_domain_top


# 6 celle per il barilotto di void:
#0. and 1.  = inner part, top and bottom, 2.  = lateral clad, 3. and 4.  = clad, top and bottom, 5. = lateral void
cellid += 1
void_cc_0 = openmc.Cell(name = 'void cc inner bot', cell_id = cellid)
void_cc_0.fill = FillingMaterial
void_cc_0.region = -s_void_r_in & +s_void_bot_in & -s_void_sep

cellid += 1
void_cc_1 = openmc.Cell(name = 'void cc inner top', cell_id = cellid)
void_cc_1.fill = Vacuum 
void_cc_1.region = -s_void_r_in & +s_void_sep & -s_void_top_in

cellid += 1
void_cc_2 = openmc.Cell(name = 'void lateral clad', cell_id = cellid)
void_cc_2.fill = CladdingAl
void_cc_2.region = +s_void_r_in & -s_void_r_out & +s_void_bot_in & -s_void_top_in

cellid += 1
void_cc_3 = openmc.Cell(name = 'void bot clad', cell_id = cellid)
void_cc_3.fill = CladdingAl
void_cc_3.region = +s_void_bot_out & -s_void_bot_in & -s_void_r_out

cellid += 1
void_cc_4 = openmc.Cell(name = 'void top vlad', cell_id = cellid)
void_cc_4.fill = CladdingAl
void_cc_4.region = +s_void_top_in & -s_void_top_out & -s_void_r_out

cellid += 1
void_cc_5 = openmc.Cell(name = 'void lateral vacuum', cell_id = cellid)
void_cc_5.fill = Vacuum
void_cc_5.region = +s_void_bot_out & -s_void_top_out & +s_void_r_out & -r_central_channel


central_channel_universe.add_cell(central_channel_bot_cell)
central_channel_universe.add_cell(central_channel_top_cell)
central_channel_universe.add_cell(central_channel_clad)
central_channel_universe.add_cell(central_channel_water)
central_channel_universe.add_cell(void_cc_0)
central_channel_universe.add_cell(void_cc_1)
central_channel_universe.add_cell(void_cc_2)
central_channel_universe.add_cell(void_cc_3)
central_channel_universe.add_cell(void_cc_4)
central_channel_universe.add_cell(void_cc_5)


#if inline_plot:
#    central_channel_universe.plot(origin=(0,0,27), width=(10.0, 90.0), pixels=[400, 400], basis='xz', color_by='material')



In [ ]:
# Original CLEAN cell 15 (zero-based)
# TRANS rod
transient_universe = openmc.Universe(name='TRANS Universe')

cellid += 1
TRANS_cell = openmc.Cell(name='TRANS Cell', cell_id = cellid)
TRANS_cell.fill = ControlRodBorated #true
TRANS_cell.region = -r_trans_inner & +bot_trans & -top_trans 

cellid += 1
TRANS_clad = openmc.Cell(name='TRANS Clad Cell', cell_id = cellid)
TRANS_clad.fill = CladdingCR
TRANS_clad.region = -r_trans_outer & +r_trans_inner & +bot_trans & -top_trans

cellid += 1
TRANS_bot_clad_cell = openmc.Cell(name='TRANS clad bot cell', cell_id = cellid)
TRANS_bot_clad_cell.fill = CladdingCR
TRANS_bot_clad_cell.region = -r_trans_outer & +bot_trans_clad & -bot_trans

cellid += 1
TRANS_bot_plug_cell = openmc.Cell(name='TRANS clad bot plug', cell_id = cellid)
TRANS_bot_plug_cell.fill = CladdingCR
TRANS_bot_plug_cell.region = -r_trans_bot_plug & +bot_trans_plug & -bot_trans_clad

cellid += 1
TRANS_top_clad_cell = openmc.Cell(name='TRANS clad top cell', cell_id = cellid)
TRANS_top_clad_cell.fill = CladdingCR
TRANS_top_clad_cell.region = -r_trans_outer & +top_trans & -top_clad_trans

cellid += 1
TRANS_water_bot = openmc.Cell(name='TRANS water Cell', cell_id = cellid)
TRANS_water_bot.fill = Water
TRANS_water_bot.region = -r_clad_103 & +bottom_grid_top & -bot_trans_plug

cellid += 1
TRANS_water_bot_plug = openmc.Cell(name='TRANS water Cell bot plug', cell_id = cellid)
TRANS_water_bot_plug.fill = Water
TRANS_water_bot_plug.region = -r_clad_103 & +r_trans_bot_plug & -bot_trans_clad & +bot_trans_plug

cellid += 1
TRANS_water_top = openmc.Cell(name='TRANS water Cell', cell_id = cellid)
TRANS_water_top.fill = Water
TRANS_water_top.region = -r_clad_103 & +top_clad_trans & -end_domain_top

cellid += 1
TRANS_water_fill = openmc.Cell(name='TRANS water Cell for filling lattice', cell_id = cellid)
TRANS_water_fill.fill = Water
TRANS_water_fill.region = +r_trans_outer & -r_clad_103 & +bot_trans_clad & -top_clad_trans


transient_universe.add_cell(TRANS_cell)
transient_universe.add_cell(TRANS_clad)
transient_universe.add_cell(TRANS_water_top)
transient_universe.add_cell(TRANS_water_bot)
transient_universe.add_cell(TRANS_water_bot_plug)
transient_universe.add_cell(TRANS_water_fill)
transient_universe.add_cell(TRANS_top_clad_cell)
transient_universe.add_cell(TRANS_bot_clad_cell)
transient_universe.add_cell(TRANS_bot_plug_cell)

#if inline_plot:
#    transient_universe.plot(origin=(0,0,z0_trans), width=(10.0, 2*z0_trans), pixels=[400, 400], basis='xz', color_by='material')



In [ ]:
# Original CLEAN cell 16 (zero-based)
# REG rod
reg_universe = openmc.Universe(name='REG Universe')

cellid += 1 
REG_cell = openmc.Cell(name='REG Cell', cell_id = cellid)
REG_cell.fill = ControlRodB4C
REG_cell.region = -r_reg_inner & +bot_reg & -top_reg 

cellid += 1
REG_clad = openmc.Cell(name='REG Clad Cell', cell_id = cellid)
REG_clad.fill = CladdingCR
REG_clad.region = -r_reg_outer & +r_reg_inner & +bot_reg & -top_reg

cellid += 1
REG_bot_clad_cell = openmc.Cell(name='REG clad bot cell', cell_id = cellid)
REG_bot_clad_cell.fill = CladdingCR
REG_bot_clad_cell.region = -r_reg_outer & +bot_reg_clad & -bot_reg

cellid += 1
REG_bot_plug_cell = openmc.Cell(name='REG clad bot plug', cell_id = cellid)
REG_bot_plug_cell.fill = CladdingCR
REG_bot_plug_cell.region = -r_reg_bot_plug & +bot_reg_plug & -bot_reg_clad

cellid += 1
REG_top_clad_cell = openmc.Cell(name='REG clad top cell', cell_id = cellid)
REG_top_clad_cell.fill = CladdingCR
REG_top_clad_cell.region = -r_reg_outer & +top_reg & -top_clad_reg

cellid += 1
REG_water_top = openmc.Cell(name='REG water Cell', cell_id = cellid)
REG_water_top.fill = Water
REG_water_top.region = -r_clad_103 & +top_clad_reg & -end_domain_top

cellid += 1
REG_water_bot = openmc.Cell(name='REG water Cell', cell_id = cellid)
REG_water_bot.fill = Water
REG_water_bot.region = -r_clad_103 & +bottom_grid_top & -bot_reg_plug

cellid += 1
REG_water_bot_plug = openmc.Cell(name='REG water Cell plug', cell_id = cellid)
REG_water_bot_plug.fill = Water
REG_water_bot_plug.region = -r_clad_103 & +r_reg_bot_plug & -bot_reg_clad & +bot_reg_plug

cellid += 1
REG_water_fill = openmc.Cell(name='REG water Cell for filling lattice', cell_id = cellid)
REG_water_fill.fill = Water
REG_water_fill.region = +r_reg_outer & -r_clad_103 & +bot_reg_clad & -top_clad_reg


reg_universe.add_cell(REG_cell)
reg_universe.add_cell(REG_clad)
reg_universe.add_cell(REG_water_top)
reg_universe.add_cell(REG_water_bot)
reg_universe.add_cell(REG_water_bot_plug)
reg_universe.add_cell(REG_water_fill)
reg_universe.add_cell(REG_top_clad_cell)
reg_universe.add_cell(REG_bot_clad_cell)
reg_universe.add_cell(REG_bot_plug_cell)


#if inline_plot:
 #   reg_universe.plot(origin=(0,0,z0_reg), width=(10, 2*z0_reg), pixels=[400, 400], basis='xz', color_by='material')



In [ ]:
# Original CLEAN cell 17 (zero-based)
# SHIM rod
shim_universe = openmc.Universe(name='SHIM Universe')

cellid += 1
SHIM_cell = openmc.Cell(name='SHIM Cell', cell_id=cellid)
SHIM_cell.fill = ControlRodB4C
SHIM_cell.region = -r_shim_inner & +bot_shim & -top_shim 

cellid += 1
SHIM_clad = openmc.Cell(name='SHIM Clad Cell', cell_id=cellid)
SHIM_clad.fill = CladdingCR
SHIM_clad.region = -r_shim_outer & +r_shim_inner & +bot_shim & -top_shim

cellid += 1
SHIM_bot_clad_cell = openmc.Cell(name='SHIM clad bot cell', cell_id=cellid)
SHIM_bot_clad_cell.fill = CladdingCR
SHIM_bot_clad_cell.region = -r_shim_outer & +bot_shim_clad & -bot_shim

cellid += 1
SHIM_bot_plug_cell = openmc.Cell(name='SHIM clad bot plug', cell_id=cellid)
SHIM_bot_plug_cell.fill = CladdingCR
SHIM_bot_plug_cell.region = -r_shim_bot_plug & +bot_shim_plug & -bot_shim_clad

cellid += 1
SHIM_top_clad_cell = openmc.Cell(name='SHIM clad top cell', cell_id=cellid)
SHIM_top_clad_cell.fill = CladdingCR
SHIM_top_clad_cell.region = -r_shim_outer & +top_shim & -top_clad_shim


cellid += 1
SHIM_water_top = openmc.Cell(name='SHIM water Cell top', cell_id=cellid)
SHIM_water_top.fill = Water
SHIM_water_top.region = -r_clad_103 & +top_clad_shim & -end_domain_top

cellid += 1
SHIM_water_bot = openmc.Cell(name='SHIM water Cell bot', cell_id=cellid)
SHIM_water_bot.fill = Water
SHIM_water_bot.region = -r_clad_103 & +end_domain_bot & -bot_shim_plug

cellid += 1
SHIM_water_bot_plug = openmc.Cell(name='SHIM water Cell bot plug', cell_id=cellid)
SHIM_water_bot_plug.fill = Water
SHIM_water_bot_plug.region = -r_clad_103 & +r_shim_bot_plug  & +bot_shim_plug & -bot_shim_clad

cellid += 1
SHIM_water_fill = openmc.Cell(name='SHIM water Cell for filling lattice', cell_id=cellid)
SHIM_water_fill.fill = Water
SHIM_water_fill.region = +r_shim_outer & -r_clad_103 & +bot_shim_clad & -top_clad_shim

shim_universe.add_cell(SHIM_cell)
shim_universe.add_cell(SHIM_clad)
shim_universe.add_cell(SHIM_water_top)
shim_universe.add_cell(SHIM_water_bot)
shim_universe.add_cell(SHIM_water_bot_plug)
shim_universe.add_cell(SHIM_water_fill)
shim_universe.add_cell(SHIM_top_clad_cell)
shim_universe.add_cell(SHIM_bot_clad_cell)
shim_universe.add_cell(SHIM_bot_plug_cell)


#if inline_plot:
 #   shim_universe.plot(origin=(0,0,z0_shim), width=(10, 2*z0_shim), pixels=[400, 400], basis='xz', color_by='material')



In [ ]:
# Original CLEAN cell 18 (zero-based)
# Dummy channel (graphite) 
graphite_channel_universe = openmc.Universe(name='Graphite Channel Universe')

cellid += 1
graphite_channel_cell = openmc.Cell(name='Graphite channel Cell', cell_id=cellid)
graphite_channel_cell.fill = Graphite
graphite_channel_cell.region = -r_dummy_graphite & +reflector_bot_101 & -reflector_top_101

cellid += 1
graphite_clad = openmc.Cell(name='Graphite Clad Cell', cell_id=cellid)
graphite_clad.fill = CladdingCR
graphite_clad.region = -r_dummy_clad & +r_dummy_graphite & +reflector_bot_101 & -reflector_top_101

cellid += 1
dummy_top_plug_cell = openmc.Cell(name='Graphite channel Cell plug top', cell_id=cellid)
dummy_top_plug_cell.fill = CladdingCR
dummy_top_plug_cell.region = -r_dummy_clad & +reflector_top_101 & -clad_top_101

cellid += 1
dummy_bot_plug_cell = openmc.Cell(name='Graphite channel Cell plug bot', cell_id=cellid)
dummy_bot_plug_cell.fill = CladdingCR
dummy_bot_plug_cell.region = -r_dummy_clad & +clad_bot_101 & -reflector_bot_101

cellid += 1
dummy_bot_water_cell = openmc.Cell(cell_id=cellid)
dummy_bot_water_cell.fill = Water
dummy_bot_water_cell.region = -r_dummy_clad & -clad_bot_101 & +end_domain_bot

cellid += 1
dummy_top_water_cell = openmc.Cell(cell_id=cellid)
dummy_top_water_cell.fill = Water
dummy_top_water_cell.region = -r_dummy_clad & +clad_top_101 & -end_domain_top


graphite_channel_universe.add_cell(graphite_channel_cell)
graphite_channel_universe.add_cell(graphite_clad)
graphite_channel_universe.add_cell(dummy_top_plug_cell)
graphite_channel_universe.add_cell(dummy_bot_plug_cell)
graphite_channel_universe.add_cell(dummy_top_water_cell)
graphite_channel_universe.add_cell(dummy_bot_water_cell)


#if inline_plot:
 #   graphite_channel_universe.plot(origin=(0,0,30), width=(10, 70), pixels=[400, 400], basis='xz', color_by='material')



In [ ]:
# Original CLEAN cell 19 (zero-based)
# Rabbit & Source empty channels
rabbit_universe = openmc.Universe(name='Rabbit Universe')

cellid += 1
rabbit_cell = openmc.Cell(name='Rabbit', cell_id=cellid)
rabbit_cell.fill = Vacuum
rabbit_cell.region = -r_rabbit_inner & +bottom_grid_top & -end_domain_top

cellid += 1
rabbit_clad = openmc.Cell(name='Rabbit clad', cell_id=cellid)
rabbit_clad.fill = CladdingCR
rabbit_clad.region = +r_rabbit_inner & -r_rabbit_outer & +bottom_grid_top & -end_domain_top

rabbit_universe.add_cell(rabbit_cell)
rabbit_universe.add_cell(rabbit_clad)



In [ ]:
# Original CLEAN cell 20 (zero-based)
# Reflector 
reflector_universe = openmc.Universe(name='Reflector')

# Create reflector cell
# --> 4 void zones inside the reflector: 1 cylindrical annulus and 3 channels

# Cylindrical ring between 31.35 and 57
cellid += 1
reflector_cell_down = openmc.Cell(name='Reflector cell down', cell_id=cellid)
reflector_cell_down.fill = Graphite
reflector_cell_down.region = -outer_radius_reflector & +inner_radius_reflector & +bot_reflector & -refl_void_b & +channel_1_outer & +channel_2_outer & +channel_3_outer #aggiungere canali

cellid += 1
reflector_cell_fill_ch1 = openmc.Cell(name='Reflector cell fill ch1', cell_id=cellid)
reflector_cell_fill_ch1.fill = Graphite
reflector_cell_fill_ch1.region = -outer_radius_reflector & +inner_radius_reflector & +bot_reflector & -refl_void_b & -channel_1_outer & -channel_1_discriminator

cellid += 1
reflector_cell_fill_ch2 = openmc.Cell(name='Reflector cell fill ch2', cell_id=cellid)
reflector_cell_fill_ch2.fill = Graphite
reflector_cell_fill_ch2.region = -outer_radius_reflector & +inner_radius_reflector & +bot_reflector & -refl_void_b & -channel_2_outer & +channel_2_discriminator

cellid += 1
reflector_cell_fill_ch3 = openmc.Cell(name='Reflector cell fill ch3', cell_id=cellid)
reflector_cell_fill_ch3.fill = Graphite
reflector_cell_fill_ch3.region = -outer_radius_reflector & +inner_radius_reflector & +bot_reflector & -refl_void_b & -channel_3_outer & +channel_3_discriminator & +channel_2_outer

cellid += 1
reflector_cell_right_vacuum = openmc.Cell(name='Reflector right side from vacuum', cell_id=cellid)
reflector_cell_right_vacuum.fill = Graphite
reflector_cell_right_vacuum.region = -refl_void_inner & +inner_radius_reflector & +refl_void_b & -top_reflector

cellid += 1
reflector_cell_left_vacuum = openmc.Cell(name='Reflector left side from vacuum', cell_id=cellid)
reflector_cell_left_vacuum.fill = Graphite
reflector_cell_left_vacuum.region = +refl_void_outer & -outer_radius_reflector & +refl_void_b & -top_reflector 

# lazy susan
cellid += 1
reflector_cell_vacuum = openmc.Cell(name='Reflector vacuum', cell_id=cellid)
reflector_cell_vacuum.fill = Vacuum
reflector_cell_vacuum.region = -refl_void_outer & +refl_void_inner & +refl_void_b & -top_reflector #aggiungere canali

cellid += 1
reflector_clad_bot = openmc.Cell(name='Reflector clad bot', cell_id=cellid)
reflector_clad_bot.fill = CladdingCR
reflector_clad_bot.region = +core_radius & -s1 & +bot_clad_reflector & -bot_reflector

cellid += 1
reflector_clad_top = openmc.Cell(name='Reflector clad top', cell_id=cellid)
reflector_clad_top.fill = CladdingCR
reflector_clad_top.region = +core_radius & -s1 & +top_reflector & -top_clad_reflector

cellid += 1
reflector_clad_side_inner = openmc.Cell(name='Reflector clad side inner', cell_id=cellid)
reflector_clad_side_inner.fill = CladdingCR
reflector_clad_side_inner.region = +core_radius & -inner_radius_reflector & +bot_reflector & -top_reflector

cellid += 1
reflector_clad_side_outer = openmc.Cell(name='Reflector clad side outer', cell_id=cellid)
reflector_clad_side_outer.fill = CladdingCR
reflector_clad_side_outer.region = +outer_radius_reflector & -s1 & +bot_reflector & -top_reflector


#CHANNELS
cellid += 1
reflector_ch1 = openmc.Cell(name='irradiation channel 1', cell_id=cellid)
reflector_ch1.fill = Vacuum
reflector_ch1.region = -channel_1_inner & +inner_radius_reflector & -outer_radius_reflector & +channel_1_discriminator

cellid += 1
reflector_ch1_clad = openmc.Cell(name='clad irradiation channel 1', cell_id=cellid)
reflector_ch1_clad.fill = CladdingCR
reflector_ch1_clad.region = +channel_1_inner & -channel_1_outer & +inner_radius_reflector & -outer_radius_reflector & +channel_1_discriminator

cellid += 1
reflector_ch2 = openmc.Cell(name='irradiation channel 2', cell_id=cellid)
reflector_ch2.fill = Vacuum
reflector_ch2.region = -channel_2_inner & +inner_radius_reflector & -outer_radius_reflector & -channel_2_discriminator

cellid += 1
reflector_ch2_clad = openmc.Cell(name='clad irradiation channel 2', cell_id=cellid)
reflector_ch2_clad.fill = CladdingCR
reflector_ch2_clad.region = +channel_2_inner & -channel_2_outer & +inner_radius_reflector & -outer_radius_reflector & -channel_2_discriminator

cellid += 1
reflector_ch3 = openmc.Cell(name='irradiation channel 3', cell_id=cellid)
reflector_ch3.fill = Vacuum
reflector_ch3.region = -channel_3_inner & +channel_2_outer & -outer_radius_reflector & -channel_3_discriminator

cellid += 1
reflector_ch3_clad = openmc.Cell(name='clad irradiation channel 3', cell_id=cellid)
reflector_ch3_clad.fill = CladdingCR
reflector_ch3_clad.region = +channel_3_inner & -channel_3_outer & +channel_2_outer & -outer_radius_reflector & -channel_3_discriminator

cellid += 1
water_reflector_top= openmc.Cell(cell_id=cellid)
water_reflector_top.region= +top_clad_reflector & -end_domain_top & -s1 & +core_radius
water_reflector_top.fill = Water

cellid += 1
water_reflector_bottom= openmc.Cell(cell_id=cellid)
water_reflector_bottom.region= +end_domain_bot & -bottom_grid_bot & -s1
water_reflector_bottom.fill = Water

cellid += 1
water_reflector_bottom2= openmc.Cell(cell_id=cellid)
water_reflector_bottom2.region= -bot_clad_reflector & +bottom_grid_bot & -s1 & +core_radius
water_reflector_bottom2.fill = Water


reflector_universe.add_cell(reflector_cell_down)
reflector_universe.add_cell(reflector_cell_vacuum)
reflector_universe.add_cell(reflector_cell_right_vacuum)
reflector_universe.add_cell(reflector_cell_left_vacuum)
reflector_universe.add_cell(reflector_clad_bot)
reflector_universe.add_cell(reflector_clad_top)
reflector_universe.add_cell(reflector_clad_side_inner)
reflector_universe.add_cell(reflector_clad_side_outer)
reflector_universe.add_cell(reflector_ch1)
reflector_universe.add_cell(reflector_cell_fill_ch1)
reflector_universe.add_cell(reflector_ch1_clad)
reflector_universe.add_cell(reflector_ch2)
reflector_universe.add_cell(reflector_cell_fill_ch2)
reflector_universe.add_cell(reflector_ch2_clad)
reflector_universe.add_cell(reflector_ch3)
reflector_universe.add_cell(reflector_cell_fill_ch3)
reflector_universe.add_cell(reflector_ch3_clad)
reflector_universe.add_cell(water_reflector_top)
reflector_universe.add_cell(water_reflector_bottom)
reflector_universe.add_cell(water_reflector_bottom2)


#if inline_plot:
 #   reflector_universe.plot(origin=(0,0,25), width=(130.0, 130), pixels=[500, 500], basis='xz', color_by='material')
  #  reflector_universe.plot(origin=(0,0,25), width=(130.0, 130), pixels=[500, 500], basis='xy', color_by='material')



In [ ]:
# Original CLEAN cell 21 (zero-based)
# --- 1965 empty water-filled slot universe ---
cellid += 1

empty_slot_water_cell = openmc.Cell(
    name="1965 empty slot water",
    fill=Water
)

empty_slot_universe = openmc.Universe(
    name="1965 Empty Slot Universe",
    cells=[empty_slot_water_cell]
)

print("empty_slot_universe created:", empty_slot_universe.name)



In [ ]:
# Original CLEAN cell 22 (zero-based)
# Core universe — ORIGINAL 1965 CONFIGURATION

# Core loading reconstructed from the original 1965 configuration:
#   61 Fuel Element 101
#   23 graphite dummy elements
#   SHIM        -> C3
#   TRANSIENT   -> D10
#   REGULATING  -> E21
#   Source      -> F4
#   Empty slot  -> F11
#   Rabbit      -> F24
#
# NOTE:
# - F4 is represented using the existing "Rabbit & Source empty channels"
#   universe already present in the base model. The detailed neutron-source
#   material itself is therefore not explicitly modeled here.
# - F11 is represented as a water-filled empty core slot.
# - All fuel positions use the generic Fuel 101 composition already defined
#   in this notebook; historical FE serial numbers / element-specific masses
#   are not required for this geometrical proof of the 1965 loading.

# Create surfaces that divide the circular core into rings
ring_radii = np.array([0.0, 4.0975, 8.2, 12.2, 16.3, 20.3])
radial_surf = [
    openmc.ZCylinder(r=r)
    for r in (ring_radii[:-1] + ring_radii[1:]) / 2
]

water_cells_down = []
water_cells_up = []
top_grid_cells = []

for i in range(ring_radii.size):

    if i == 0:
        water_region_down = -radial_surf[i] & -top_grid_bot
        water_region_up = -radial_surf[i] & +top_grid_top
        grid_region = -radial_surf[i] & +top_grid_bot & -top_grid_top

    elif i == ring_radii.size - 1:
        water_region_down = -top_grid_bot & +radial_surf[i-1] & -core_radius
        water_region_up = +top_grid_top & +radial_surf[i-1] & -core_radius
        grid_region = +radial_surf[i-1] & +top_grid_bot & -top_grid_top

    else:
        water_region_down = (
            -top_grid_bot & +radial_surf[i-1] & -radial_surf[i]
        )
        water_region_up = (
            +top_grid_top & +radial_surf[i-1] & -radial_surf[i]
        )
        grid_region = (
            +radial_surf[i-1] & -radial_surf[i]
            & +top_grid_bot & -top_grid_top
        )

    cellid += 1
    water_cell_down = openmc.Cell(
        fill=Water, region=water_region_down, cell_id=cellid
    )

    cellid += 1
    water_cell_up = openmc.Cell(
        fill=Water, region=water_region_up, cell_id=cellid
    )

    cellid += 1
    top_grid_cell = openmc.Cell(
        fill=CladdingAl, region=grid_region, cell_id=cellid
    )

    water_cells_down.append(water_cell_down)
    water_cells_up.append(water_cell_up)
    top_grid_cells.append(top_grid_cell)


core_universe = openmc.Universe()
core_universe.add_cells(water_cells_down)
core_universe.add_cells(water_cells_up)
core_universe.add_cells(top_grid_cells)


# -------------------------------------------------------------------------
# Ring convention
# -------------------------------------------------------------------------
# Ring A: central channel
# Ring B:  6 positions
# Ring C: 12 positions
# Ring D: 18 positions
# Ring E: 24 positions
# Ring F: 30 positions
#
# The function below converts the (ring index, OpenMC angular index j)
# used by the original notebook into the familiar labels B1...F30.
# This is the same orientation used for the 2013 mapping.

num_pins = [1, 6, 12, 18, 24, 30]
angles = [0, 0, 0, 0, 0, 0]
ring_letters = ["A", "B", "C", "D", "E", "F"]


def core_position(i, j, n):
    if i == 0:
        return "CC"
    position_number = ((i - j) % n) + 1
    return f"{ring_letters[i]}{position_number}"


# -------------------------------------------------------------------------
# Explicit 1965 loading map
# -------------------------------------------------------------------------

fuel_101_positions = set(
    [f"B{k}" for k in range(1, 7)]
    + [f"C{k}" for k in range(1, 13) if k != 3]
    + [f"D{k}" for k in range(1, 19) if k != 10]
    + [f"E{k}" for k in range(1, 25) if k != 21]
    + [f"F{k}" for k in range(25, 29)]
)

graphite_positions = set(
    [f"F{k}" for k in [1, 2, 3]]
    + [f"F{k}" for k in range(5, 11)]
    + [f"F{k}" for k in range(12, 24)]
    + [f"F{k}" for k in [29, 30]]
)

shim_positions = {"C3"}
transient_positions = {"D10"}
regulating_positions = {"E21"}

source_positions = {"F4"}
empty_positions = {"F11"}
rabbit_positions = {"F24"}


# -------------------------------------------------------------------------
# Consistency checks BEFORE creating the geometry
# -------------------------------------------------------------------------

all_BF_positions = (
    {f"B{k}" for k in range(1, 7)}
    | {f"C{k}" for k in range(1, 13)}
    | {f"D{k}" for k in range(1, 19)}
    | {f"E{k}" for k in range(1, 25)}
    | {f"F{k}" for k in range(1, 31)}
)

categories = [
    fuel_101_positions,
    graphite_positions,
    shim_positions,
    transient_positions,
    regulating_positions,
    source_positions,
    empty_positions,
    rabbit_positions,
]

assigned_positions = set().union(*categories)

# Check that categories do not overlap
sum_category_lengths = sum(len(s) for s in categories)
if len(assigned_positions) != sum_category_lengths:
    raise RuntimeError("1965 loading map contains overlapping assignments.")

# Check that every B-F core position is assigned exactly once
if assigned_positions != all_BF_positions:
    missing = sorted(all_BF_positions - assigned_positions)
    extra = sorted(assigned_positions - all_BF_positions)
    raise RuntimeError(
        f"1965 loading map incomplete. Missing={missing}, Extra={extra}"
    )

# Historical loading counts from the reference configuration
if len(fuel_101_positions) != 61:
    raise RuntimeError(
        f"Expected 61 FE101, found {len(fuel_101_positions)}."
    )

if len(graphite_positions) != 23:
    raise RuntimeError(
        f"Expected 23 graphite dummies, found {len(graphite_positions)}."
    )

print("1965 loading map checks:")
print("  Fuel 101       =", len(fuel_101_positions))
print("  Graphite dummy =", len(graphite_positions))
print("  SHIM           =", shim_positions)
print("  TRANSIENT      =", transient_positions)
print("  REGULATING     =", regulating_positions)
print("  Source         =", source_positions)
print("  Empty slot     =", empty_positions)
print("  Rabbit         =", rabbit_positions)
print("  Total B-F slots=", len(assigned_positions))
print("1965 LOADING MAP CHECK OK")


# -------------------------------------------------------------------------
# Fill the core
# -------------------------------------------------------------------------

for i, (r, n, a) in enumerate(zip(ring_radii, num_pins, angles)):

    for j in range(n):

        theta = (a + j / n * 360.0) * np.pi / 180.0
        x = r * np.cos(theta)
        y = r * np.sin(theta)

        position = core_position(i, j, n)

        cellid += 1

        pin_boundary = openmc.ZCylinder(
            x0=x, y0=y, r=r_clad_101.r
        )
        pin_boundary_CC = openmc.ZCylinder(
            x0=x, y0=y, r=r_central_channel_clad.r
        )

        # Preserve the original model treatment of the water/grid regions
        water_cells_down[i].region &= +pin_boundary & -core_radius
        water_cells_up[i].region &= +pin_boundary & -core_radius
        top_grid_cells[i].region &= +pin_boundary & -core_radius

        if position == "CC":
            print("Adding central channel...")
            pin = openmc.Cell(
                fill=central_channel_universe,
                region=-pin_boundary_CC,
                cell_id=cellid,
            )

        elif position in transient_positions:
            print(f"{position}: adding TRANSIENT rod...")
            pin = openmc.Cell(
                fill=transient_universe,
                region=-pin_boundary,
                cell_id=cellid,
            )

        elif position in shim_positions:
            print(f"{position}: adding SHIM rod...")
            pin = openmc.Cell(
                fill=shim_universe,
                region=-pin_boundary,
                cell_id=cellid,
            )

        elif position in regulating_positions:
            print(f"{position}: adding REGULATING rod...")
            pin = openmc.Cell(
                fill=reg_universe,
                region=-pin_boundary,
                cell_id=cellid,
            )

        elif position in source_positions:
            print(f"{position}: adding source channel (simplified)...")
            pin = openmc.Cell(
                fill=rabbit_universe,
                region=-pin_boundary,
                cell_id=cellid,
            )

        elif position in rabbit_positions:
            print(f"{position}: adding rabbit channel...")
            pin = openmc.Cell(
                fill=rabbit_universe,
                region=-pin_boundary,
                cell_id=cellid,
            )

        elif position in empty_positions:
            print(f"{position}: adding water-filled empty slot...")
            pin = openmc.Cell(
                fill=empty_slot_universe,
                region=-pin_boundary,
                cell_id=cellid,
            )
        elif position in graphite_positions:
            print(f"{position}: adding graphite dummy...")
            pin = openmc.Cell(
                fill=graphite_channel_universe,
                region=-pin_boundary,
                cell_id=cellid,
            )

        elif position in fuel_101_positions:
            print(f"{position}: adding Fuel 101...")

            # Per ora TUTTE le FE101, inclusa B1, usano l'universo normale
            pin = openmc.Cell(
                fill=fuel_101_universe,
                region=-pin_boundary,
                cell_id=cellid,
            )

            # Conserviamo solo il riferimento alla cella fisica B1
            if position == "B1":
                B1_pin = pin

        else:
            raise RuntimeError(
                f"No 1965 loading assignment for {position}."
            )

        pin.translation = (x, y, 0)
        pin.id = (i + 3) * 100 + j
        core_universe.add_cell(pin)

        


# -------------------------------------------------------------------------
# Quick geometry plots
# -------------------------------------------------------------------------

 #if inline_plot
 # core_universe.plot(
       # origin=(0, 0, 25),
          #width=(60.0, 100),
        #pixels=[500, 500],
        #basis="xz",
        #color_by="material",
        #seed=3,
    #)

#if inline_plot:
    #core_universe.plot(
     #   origin=(0, 0, 25),
     #   width=(60.0, 60),
      #  pixels=[500, 500],
       # basis="xy",
        #color_by="material",
        #seed=2,
    #)




In [ ]:
# Original CLEAN cell 26 (zero-based)
# Geometry definitions for the reactor

cellid += 1
reactor = openmc.Cell(cell_id = cellid)
reactor.region = -core_radius & -end_domain_top & +bottom_grid_top
reactor.fill = core_universe

cellid += 1
reflector = openmc.Cell(cell_id = cellid)
reflector.region = +core_radius & -s1 & +end_domain_bot & -end_domain_top
reflector.fill = reflector_universe

cellid += 1
bottom_grid_cell = openmc.Cell(name='bottom grid cell', cell_id = cellid)
bottom_grid_cell.fill = CladdingCR
bottom_grid_cell.region = -core_radius & +bottom_grid_bot & -bottom_grid_top

cellid += 1
bottom_grid_cell_water = openmc.Cell(name='bottom grid cell', cell_id = cellid)
bottom_grid_cell_water.fill = Water
bottom_grid_cell_water.region = -core_radius & -bottom_grid_bot & +end_domain_bot


### --- ROOT UNIVERSE   
triga_universe = openmc.Universe(name='TRIGA universe')

triga_universe.add_cell(reactor)
triga_universe.add_cell(reflector)
triga_universe.add_cell(bottom_grid_cell)
triga_universe.add_cell(bottom_grid_cell_water)


geometry = openmc.Geometry((triga_universe))

#if inline_plot:
 #   triga_universe.plot(origin=(0,0,25), width=(130.0, 130), pixels=[700, 700], basis='xz', color_by='material', seed = 3)

#if inline_plot:
 #   triga_universe.plot(origin=(0,0,25), width=(130.0, 130), pixels=[700, 700], basis='xy', color_by='material', seed = 3)



## Dedicated physical B1: five rings and preparation audit


In [ ]:
"""Notebook preparation cell: five rings in B1, physical gap, no transport."""
import json

Fuel101.depletable = False
Fuel103.depletable = False
R_FUEL_CM = 1.791
R_CLAD_INNER_CM = 1.804
R_CLAD_OUTER_CM = 1.880
ring_edges_cm = R_FUEL_CM * np.sqrt(np.arange(6) / 5.0)
active_height_cm = top_active_101.z0 - bot_active_101.z0
original_fuel_volume_cm3 = np.pi * R_FUEL_CM**2 * active_height_cm

# Clone the target universe only; shared ordinary FE101 regions are untouched.
fuel_101_universe_B1 = fuel_101_universe.clone(
    clone_materials=False, clone_regions=False)
fuel_101_universe_B1.name = "FE101 B1 physical geometry B - five radial rings"
old_target_cell = next(c for c in fuel_101_universe_B1.cells.values()
                       if c.name == "Fuel 101")
fuel_101_universe_B1.remove_cell(old_target_cell)
active_region = +bot_active_101 & -top_active_101
ring_surfaces = [None] + [openmc.ZCylinder(r=float(r)) for r in ring_edges_cm[1:-1]] + [r_fuel_101]
ring_materials, ring_cells = [], []
for i in range(5):
    material = Fuel101.clone()
    material.name = f"Fuel101_B1_ring_{i+1}"
    material.depletable = True
    material.volume = float(np.pi * (ring_edges_cm[i+1]**2-ring_edges_cm[i]**2)
                            * active_height_cm)
    region = -ring_surfaces[i+1] & active_region
    if i > 0:
        region = region & +ring_surfaces[i]
    cell = openmc.Cell(name=f"FE101 B1 active ring {i+1}", fill=material, region=region)
    ring_materials.append(material)
    ring_cells.append(cell)
    fuel_101_universe_B1.add_cell(cell)
    materials_file.append(material)

target_clad = next(c for c in fuel_101_universe_B1.cells.values()
                   if c.name == "Al Cladding")
physical_clad_inner = openmc.ZCylinder(r=R_CLAD_INNER_CM)
# Preserve all axial/end regions; only active fuel-gap-clad changes.
target_clad.region = target_clad.region & (
    +physical_clad_inner | -bot_active_101 | +top_active_101)
gap_helium = openmc.Material(name="FE101 B1 negligible-density He4 gap")
gap_helium.add_nuclide("He4", 1.0)
gap_helium.set_density("g/cm3", 1.0e-10)
gap_helium.temperature = T_ref_struct
materials_file.append(gap_helium)
target_gap = openmc.Cell(name="FE101 B1 active 130 um He gap", fill=gap_helium,
                        region=+r_fuel_101 & -physical_clad_inner & active_region)
fuel_101_universe_B1.add_cell(target_gap)
B1_pin.fill = fuel_101_universe_B1

# Pure Python geometry/inventory audit. No OpenMC executable or lib.init.
geometry.determine_paths(instances_only=True)
depletable = [m for m in materials_file if m.depletable]
assert len(depletable) == 5
assert {m.id for m in depletable} == {m.id for m in ring_materials}
assert {m.id for m in geometry.get_all_materials().values() if m.depletable} == {m.id for m in ring_materials}
assert Fuel101.num_instances == 60 and not Fuel101.depletable
assert np.isclose(sum(m.volume for m in ring_materials), original_fuel_volume_cm3, rtol=1e-14)

def element_of(nuclide):
    return "".join(c for c in nuclide if c.isalpha())

reference_densities = Fuel101.get_nuclide_atom_densities()
reference_mass_g = Fuel101.get_mass(volume=original_fuel_volume_cm3)
origin = np.asarray(B1_pin.translation)
audit_rings = []
for i, (material, cell) in enumerate(zip(ring_materials, ring_cells)):
    assert material.num_instances == cell.num_instances == 1
    assert material.nuclides == Fuel101.nuclides
    assert material.density == 6.3 and material.density_units == "g/cm3"
    assert material.temperature == Fuel101.temperature
    assert material._sab == Fuel101._sab
    densities = material.get_nuclide_atom_densities()
    assert densities == reference_densities
    # OpenMC atomic densities are atom/b-cm, volume is cm3.
    initial_atoms = {n: float(d * 1e24 * material.volume) for n, d in densities.items()}
    metal_atoms = sum(v for n, v in initial_atoms.items() if element_of(n) in ("U", "Zr"))
    u_mass_kg = sum(material.get_mass(n) for n in densities if element_of(n) == "U") / 1000
    h_zr = (sum(d for n, d in densities.items() if element_of(n) == "H") /
            sum(d for n, d in densities.items() if element_of(n) == "Zr"))
    for phi in (0.0, 0.7, 2.0, 4.0):
        for z in (bot_active_101.z0+1e-6, (bot_active_101.z0+top_active_101.z0)/2,
                  top_active_101.z0-1e-6):
            radius = np.sqrt((ring_edges_cm[i]**2+ring_edges_cm[i+1]**2)/2)
            point = origin + [radius*np.cos(phi), radius*np.sin(phi), z]
            assert geometry.find(tuple(point))[-1].fill is material
    audit_rings.append({
        "ring": i+1, "material_id": material.id, "cell_id": cell.id,
        "instances": material.num_instances,
        "r_in_cm": float(ring_edges_cm[i]), "r_out_cm": float(ring_edges_cm[i+1]),
        "r_in_over_R": float(ring_edges_cm[i]/R_FUEL_CM),
        "r_out_over_R": float(ring_edges_cm[i+1]/R_FUEL_CM),
        "volume_cm3": material.volume, "density_g_cm3": material.density,
        "mass_g": material.get_mass(), "initial_U_mass_kg": u_mass_kg,
        "initial_U_Zr_atoms": metal_atoms, "initial_atoms_by_nuclide": initial_atoms,
        "H_Zr_atom_ratio": h_zr, "temperature_K": material.temperature,
    })
assert np.isclose(sum(m.get_mass() for m in ring_materials), reference_mass_g, rtol=1e-14)
for nuc, density in reference_densities.items():
    actual = sum(r["initial_atoms_by_nuclide"][nuc] for r in audit_rings)
    assert np.isclose(actual, density*1e24*original_fuel_volume_cm3, rtol=1e-14)
for radius, expected in [(R_FUEL_CM-1e-6, ring_materials[-1]),
                         (R_FUEL_CM+1e-6, gap_helium),
                         (R_CLAD_INNER_CM-1e-6, gap_helium),
                         (R_CLAD_INNER_CM+1e-6, CladdingAl)]:
    point = origin + [radius, 0, (bot_active_101.z0+top_active_101.z0)/2]
    assert geometry.find(tuple(point))[-1].fill is expected
# Ordinary FE101 cell still has the original fuel/clad regions.
assert fuel_101_cell.fill is Fuel101
assert clad_101_cell.fill is CladdingAl

audit = {
    "geometry_baseline": "physical B; legacy A is not the radial baseline",
    "gap_radial_um": (R_CLAD_INNER_CM-R_FUEL_CM)*1e4,
    "gap_nuclide": "He4", "gap_density_g_cm3": 1e-10,
    "active_z_cm": [bot_active_101.z0, top_active_101.z0],
    "B1_origin_cm": origin.tolist(), "original_volume_cm3": original_fuel_volume_cm3,
    "reconstructed_volume_cm3": sum(m.volume for m in ring_materials),
    "original_mass_g": reference_mass_g,
    "reconstructed_mass_g": sum(m.get_mass() for m in ring_materials),
    "ordinary_FE101_instances": Fuel101.num_instances,
    "depletable_materials": len(depletable), "rings": audit_rings,
    "audit_method": "Python instance counts, analytic volumes and point locations; no transport",
}
print("PASS: five distinct materials, one instance each; 60 ordinary FE101 unchanged")
print("PASS: volumes, mass and every initial nuclide inventory reconstruct the original fuel")
print("PASS: physical 130 um He gap; unchanged axial boundaries")
print(pd.DataFrame(audit_rings)[["ring", "volume_cm3", "mass_g", "initial_U_Zr_atoms",
                               "H_Zr_atom_ratio", "instances"]].to_string(index=False))


## Settings and nuclear data

Settings remain those of the CLEAN single-FE101 preliminary verification:
30 batches, 10 inactive, 5000 particles/batch. No high-statistics sensitivity
settings are silently substituted. Original temperatures, rods, source and
nuclear data are retained. The following tally definitions preserve the original
flux and global heating diagnostics and add material-resolved ring diagnostics.


In [ ]:
# Original CLEAN cell 31 (zero-based)
# OpenMC simulation parameters — preliminary 1965 verification run 

entropy_calc = False
resonance_calc = False
uniform_source = True

batches = 30
inactive = 10
particles = 5000

settings_file = openmc.Settings()
settings_file.batches = batches
settings_file.inactive = inactive
settings_file.particles = particles

# uniform initial source
if uniform_source:
    bounds = [-25, -25, 5, 25, 25, 50]
    uniform_dist = openmc.stats.Box(bounds[:3], bounds[3:])
    settings_file.source = openmc.IndependentSource(space=uniform_dist)

if resonance_calc:
    settings_file.resonance_scattering['enable'] = True

if entropy_calc:
    entropy_mesh = openmc.Mesh()
    entropy_mesh.lower_left = (-60, -60, -5)
    entropy_mesh.upper_right = (60, 60, 60)

    entropy_mesh.dimension = (10, 10, 10)
    settings_file.entropy_mesh = entropy_mesh

settings_file.temperature = {'method':'nearest', 'tolerance': 10}




In [ ]:
# Original CLEAN cell 33 (zero-based)
# Measuring quantities during simulation
tallies_file = openmc.Tallies()

### Neutron energy flux 
energies = np.logspace(np.log10(1e-5), np.log10(20.0e6), 101)
e_filter = openmc.EnergyFilter(energies)

EnergySpectrum = openmc.Tally(name = "Phi(E)")
EnergySpectrum.filters = [e_filter]
EnergySpectrum.scores = ['flux']


### Spatial flux
mesh3D = openmc.RegularMesh()
mesh3D.dimension = (60, 60, 100)
mesh3D.lower_left = (-55, -55, -10)
mesh3D.upper_right = (55, 55, 100)

mesh3D_filter = openmc.MeshFilter(mesh3D)

SpatialFlux = openmc.Tally(name = "Phi(r)")
SpatialFlux.filters = [mesh3D_filter]
SpatialFlux.scores = ['flux']


### Deposited energy (for normalization)
heating = openmc.Tally(name='factor-for-normalization')
heating.scores = ['heating-local']

tallies_file.append(EnergySpectrum)
tallies_file.append(SpatialFlux)
tallies_file.append(heating)




In [ ]:
# Original CLEAN cell 34 (zero-based)
import os
import openmc

# Supply ENDF/B-VIII.0 data explicitly; never silently select another library.
xs = os.environ.get("OPENMC_CROSS_SECTIONS") or openmc.config.get("cross_sections")
chain = os.environ.get("OPENMC_CHAIN_FILE") or openmc.config.get("chain_file")
if not xs:
    raise ValueError("Set OPENMC_CROSS_SECTIONS to the ENDF/B-VIII.0 cross_sections.xml")

os.environ["OPENMC_CROSS_SECTIONS"] = xs

openmc.config["cross_sections"] = xs
if chain:
    openmc.config["chain_file"] = chain

print("XS:", openmc.config["cross_sections"])
print("Chain:", chain)


In [ ]:
ring_filter = openmc.MaterialFilter(ring_materials)
ring_fission_tally = openmc.Tally(name="FE101 B1 ring total fission")
ring_fission_tally.filters = [ring_filter]
ring_fission_tally.scores = ["fission"]
ring_fission_tally.estimator = "tracklength"
ring_energy_tally = openmc.Tally(name="FE101 B1 ring energy deposition")
ring_energy_tally.filters = [ring_filter]
ring_energy_tally.scores = ["heating-local"]
tallies_file.extend([ring_fission_tally, ring_energy_tally])
radial_model = openmc.Model(geometry=geometry, materials=materials_file,
                            settings=settings_file, tallies=tallies_file)


In [ ]:
audit.update({
    "source_notebook": 'triga_single_FE101_B1_1965_2026_CLEAN.ipynb',
    "source_model": "../fe101_geometry_static/source_model.py",
    "source_model_sha256": '8a1a343837f6dc416c7ca96c2ac1d0a645b03e86fd5af0b55825080954ff0d78',
    "source_sha256": '7375bb5f97235160327ae10c227f1c2241ea603abbe1f6379a35eb5c0bc122a7',
    "cross_sections": xs, "chain_file": chain,
    "settings": {"batches": settings_file.batches, "inactive": settings_file.inactive,
                 "particles": settings_file.particles},
    "normalization_mode": "energy-deposition", "diff_burnable_mats": False,
    "power_W": 250000.0, "steps": 20, "step_hours": 175.0,
    "write_rates": True, "final_step": True,
    "FIMA_denominator": "fixed initial U+Zr atoms from exact input composition; H excluded",
    "FIMA_primary_for_ZEST": True,
})
if not chain:
    raise ValueError("Set OPENMC_CHAIN_FILE to chain-endf-b8.0.xml for depletion preparation")
if EXPORT_PREPARED_INPUTS:
    PREPARED_DIR.mkdir(parents=True, exist_ok=True)
    geometry.export_to_xml(PREPARED_DIR/"geometry.xml")
    materials_file.export_to_xml(PREPARED_DIR/"materials.xml")
    settings_file.export_to_xml(PREPARED_DIR/"settings.xml")
    tallies_file.export_to_xml(PREPARED_DIR/"tallies.xml")
    (PREPARED_DIR/"preparation_audit.json").write_text(json.dumps(audit, indent=2)+"\n")
    print("Prepared initial XMLs and fixed initial inventories:", PREPARED_DIR)


## Depletion — disabled until explicitly authorized

Predictor: 20 intervals × 175 h = 3500 h, at **250 kW whole-reactor power**.
Use `energy-deposition`, `diff_burnable_mats=False`, `write_rates=True`,
`final_step=True`. There will be 21 saved time points and 21 transport
evaluations, including the genuine final-rate calculation. Five ring materials
are already distinct; do not enable automatic material differentiation.

Changing `RUN_DEPLETION` is an explicit execution switch. Preparation does not
construct or initialize a depletion operator. A fresh run directory is required
to prevent accidental mixing with old one-material or prior five-ring results.


In [ ]:
if RUN_DEPLETION:
    import openmc.deplete.pool
    openmc.deplete.pool.USE_MULTIPROCESSING = False
    if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
        raise FileExistsError(f"Use a fresh depletion directory: {RUN_DIR}")
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    if not (PREPARED_DIR/"preparation_audit.json").exists():
        raise FileNotFoundError("Export preparation_audit.json before running")
    operator = openmc.deplete.CoupledOperator(
        radial_model, chain_file=openmc.config["chain_file"],
        normalization_mode="energy-deposition", diff_burnable_mats=False)
    operator.output_dir = RUN_DIR
    assert len(operator.burnable_mats) == 5
    assert set(operator.burnable_mats) == {str(m.id) for m in ring_materials}
    integrator = openmc.deplete.PredictorIntegrator(
        operator, timesteps=[175.0]*20, power=250000.0, timestep_units="h")
    integrator.integrate(
        path="depletion_results_FE101_B1_1965_5RINGS_3500h_ED.h5",
        write_rates=True, final_step=True)
else:
    print("Depletion/transport disabled: preparation only.")


## Postprocessing plan — inventories, direct FIMA and independent energetic BU

For each ring and each of 21 times, save U235/U238/Pu239/Cs137 inventories
in atoms; total physical fission rate summed over all fissioning chain nuclides;
cumulative fissions; FIMA; deposited power and energy; BU in MWd/kg initial U.

Rates read through `Results.get_reaction_rate()` are already fissions/s, not
per-atom rates. FIMA uses fixed initial U+Zr atoms, excludes H and is a fraction
(0.01 means 1% FIMA). No 200 MeV/fission conversion is used.

For this first Predictor verification, cumulative fissions and deposited energy
use first-order BOS/left-endpoint integration: sum(rate[k]*dt[k]). This is a
time-discretization approximation, not an exact integrated Bateman counter.
The final rate is real because `final_step=True`, but adds no further interval.

BU integrates the material-resolved `heating-local` tally normalized by global
heating at 250 kW and divides by fixed initial U mass. It means **energy
deposited in that ring**, with local secondary-photon deposition assumption;
it is distinct from a fission-energy-production burnup. Capture heating is
included. Neither the FE nor an individual ring is assigned the whole 250 kW.

Outputs in `run_3500h_ED/postprocessing/`:
- `ring_history.csv` and `ring_history.json`: 105 ring/time records.
- `fission_rates_by_nuclide.csv`: isotope-resolved rates used in the sum.
- `zest_FIMA.csv`: primary transfer file, with time, ring radii/r/R and FIMA.

Initial geometry, volumes, U mass and atom-count denominators are preserved in
`prepared_inputs/preparation_audit.json`. All 21 BOS/final statepoints and
the depletion H5 are retained. Statistical/time-integration uncertainties are
not silently claimed to be propagated into cumulative FIMA/BU.


In [ ]:
"""Prepared postprocessing; runs only on explicit invocation after depletion."""
def postprocess_radial(run_directory, preparation_audit):
    import json
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import openmc
    import openmc.deplete

    run_directory = Path(run_directory)
    metadata = json.loads(Path(preparation_audit).read_text())
    results_path = run_directory / "depletion_results_FE101_B1_1965_5RINGS_3500h_ED.h5"
    results = openmc.deplete.Results(results_path)
    times = np.asarray(results.get_times(time_units="s"), dtype=float)
    assert len(times) == 21 and np.all(np.diff(times) > 0)
    assert np.isclose(times[-1], 3500*3600)
    ring_metadata = metadata["rings"]
    assert set(results[0].index_mat) == {str(r["material_id"]) for r in ring_metadata}
    # Read the same chain used by the run; include fast-fission contributors,
    # e.g. U238, and actinides formed during irradiation, not only U235/Pu239.
    chain = openmc.deplete.Chain.from_xml(metadata["chain_file"])
    fission_nuclides = [n.name for n in chain.nuclides
                       if any(reaction.type == "fission" for reaction in n.reactions)]
    with openmc.StatePoint(run_directory / "openmc_simulation_n0.h5") as sp:
        score = sp.get_tally(name="FE101 B1 ring energy deposition").scores[0]
        assert score == "heating-local"
    power_by_ring = np.zeros((len(times), 5))
    tally_fission_by_ring = np.zeros_like(power_by_ring)
    # Predictor saves BOS statepoints n0..n19 and the final transport n20.
    for k in range(len(times)):
        path = run_directory / f"openmc_simulation_n{k}.h5"
        with openmc.StatePoint(path) as sp:
            global_heat = float(sp.get_tally(name="factor-for-normalization").mean.ravel()[0])
            if global_heat <= 0:
                raise ValueError(f"Non-positive global heating in {path}")
            ring_heat = sp.get_tally(name="FE101 B1 ring energy deposition")
            ring_fission = sp.get_tally(name="FE101 B1 ring total fission")
            heat_ids = list(ring_heat.filters[0].bins)
            fission_ids = list(ring_fission.filters[0].bins)
            source_per_s = 250000.0 / (global_heat * openmc.data.JOULE_PER_EV)
            for i, ring in enumerate(ring_metadata):
                mid = ring["material_id"]
                power_by_ring[k, i] = 250000.0 * ring_heat.mean[heat_ids.index(mid), 0, 0] / global_heat
                tally_fission_by_ring[k, i] = ring_fission.mean[fission_ids.index(mid), 0, 0] * source_per_s

    rows, nuclide_rate_rows, zest_rows = [], [], []
    for i, ring in enumerate(ring_metadata):
        mid = str(ring["material_id"])
        total_rate = np.zeros(len(times))
        for nuc in fission_nuclides:
            if nuc not in results[0].rates.index_nuc:
                if nuc in results[0].index_nuc:
                    _, atoms = results.get_atoms(mid, nuc)
                    if np.any(atoms > 0):
                        raise ValueError(f"Fissioning nuclide {nuc} has inventory but no saved rate")
                continue
            t, rates = results.get_reaction_rate(mid, nuc, "fission")
            assert np.allclose(t, times)
            total_rate += rates
            for k, rate in enumerate(rates):
                nuclide_rate_rows.append({"time_s": times[k], "ring": i+1,
                                          "nuclide": nuc, "fission_rate_per_s": float(rate)})
        inventory = {}
        for nuc in ("U235", "U238", "Pu239", "Cs137"):
            t, atoms = results.get_atoms(mid, nuc, nuc_units="atoms")
            assert np.allclose(t, times)
            inventory[nuc] = atoms
        for nuc in ("U235", "U238"):
            assert np.isclose(inventory[nuc][0], ring["initial_atoms_by_nuclide"][nuc], rtol=1e-10)
        # First-order left-endpoint integration, matching Predictor's BOS
        # transport sampling. Not an exact Bateman-integrated fission count.
        # Final_step=True supplies the genuine final rate but no extra interval.
        cumulative = np.r_[0.0, np.cumsum(total_rate[:-1] * np.diff(times))]
        energy_j = np.r_[0.0, np.cumsum(power_by_ring[:-1, i] * np.diff(times))]
        fima = cumulative / ring["initial_U_Zr_atoms"]
        burnup = energy_j / (8.64e10 * ring["initial_U_mass_kg"])
        for k, t in enumerate(times):
            row = {
                "time_s": t, "time_h": t/3600, "ring": i+1, "material_id": int(mid),
                "r_in_cm": ring["r_in_cm"], "r_out_cm": ring["r_out_cm"],
                "r_in_over_R": ring["r_in_over_R"], "r_out_over_R": ring["r_out_over_R"],
                "volume_cm3": ring["volume_cm3"],
                "initial_U_Zr_atoms": ring["initial_U_Zr_atoms"],
                "initial_U_mass_kg": ring["initial_U_mass_kg"],
                **{f"{n}_atoms": float(a[k]) for n, a in inventory.items()},
                "fission_rate_per_s": float(total_rate[k]),
                "tally_fission_rate_per_s": float(tally_fission_by_ring[k, i]),
                "cumulative_fissions": float(cumulative[k]),
                "FIMA": float(fima[k]), "FIMA_percent": float(100*fima[k]),
                "deposited_power_W": float(power_by_ring[k, i]),
                "cumulative_deposited_energy_J": float(energy_j[k]),
                "BU_MWd_kgU": float(burnup[k]),
            }
            rows.append(row)
            zest_rows.append({key: row[key] for key in ("time_s", "time_h", "ring", "r_in_cm", "r_out_cm",
                             "r_in_over_R", "r_out_over_R", "FIMA")})
    output = run_directory / "postprocessing"
    output.mkdir(exist_ok=True)
    pd.DataFrame(rows).sort_values(["time_s", "ring"]).to_csv(output/"ring_history.csv", index=False)
    pd.DataFrame(nuclide_rate_rows).to_csv(output/"fission_rates_by_nuclide.csv", index=False)
    pd.DataFrame(zest_rows).sort_values(["time_s", "ring"]).to_csv(output/"zest_FIMA.csv", index=False)
    payload = {
        "geometry_baseline": "physical B", "primary_ZEST_variable": "FIMA (fraction)",
        "fima_definition": "cumulative fissions / fixed initial U+Zr atoms; H excluded",
        "integration": "first-order BOS/left-endpoint; time-discretization approximation",
        "burnup_definition": "energy deposited in ring / initial uranium mass; MWd/kgU",
        "energy_normalization": "250 kW whole-reactor heating-local; no fixed energy/fission",
        "uncertainties": "Monte Carlo/depletion/time-integration errors not propagated into cumulative outputs",
        "rings": ring_metadata, "history": sorted(rows, key=lambda r: (r["time_s"], r["ring"])),
    }
    (output/"ring_history.json").write_text(json.dumps(payload, indent=2)+"\n")
    print(f"Saved {len(rows)} ring/time rows to {output}")
    return pd.DataFrame(rows)


In [ ]:
if RUN_POSTPROCESS:
    ring_history = postprocess_radial(RUN_DIR, PREPARED_DIR/"preparation_audit.json")
    display(ring_history.tail(5))
else:
    print("Postprocessing disabled until the five-ring result exists.")
